# BioHub Cell Tracking - Google Colab Fine-Tuning

This notebook fine-tunes the 50-epoch pre-trained 3D Temporal UNet and Edge Predictor on Google Colab's free T4 GPU.

In [ ]:
# Mount Google Drive to save our fine-tuned weights permanently
from google.colab import drive
import os

drive.mount('/content/drive')

# Setup Kaggle API to download the dataset (needs kaggle.json)
from google.colab import userdata
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except:
    import getpass
    print("\nPlease enter your Kaggle API credentials (found in your kaggle.json account settings):")
    os.environ['KAGGLE_USERNAME'] = getpass.getpass('KAGGLE_USERNAME: ')
    os.environ['KAGGLE_KEY'] = getpass.getpass('KAGGLE_KEY: ')


In [ ]:
# Install required dependencies for the tracking code
!pip install zarr rustworkx networkx numcodecs


In [ ]:
# Download the competition data and the official pre-trained 50-epoch weights
!mkdir -p /content/data /content/weights /content/repo

print("Downloading competition dataset...")
!kaggle competitions download -c biohub-cell-tracking-during-development -p /content/data
!unzip -q -o /content/data/biohub-cell-tracking-during-development.zip -d /content/data/

print("Downloading pre-trained weights & training code...")
!kaggle datasets download pilkwang/biohub-tracking-support-pack-50ep-v1 -p /content/weights
!unzip -q -o /content/weights/biohub-tracking-support-pack-50ep-v1.zip -d /content/weights/
!unzip -q -o /content/weights/tracking_repo.zip -d /content/repo/


In [ ]:
# Patch the training script to support --resume for the FULL model (not just UNet)
import torch

script_path = '/content/repo/scripts/train_unet_transformer.py'
with open(script_path, 'r') as f:
    lines = f.readlines()
    
for i, line in enumerate(lines):
    if 'parser.add_argument("--unet-weights"' in line:
        lines.insert(i, '    parser.add_argument("--resume", type=str, default=None, help="Full checkpoint to resume.")\n')
        break

for i, line in enumerate(lines):
    if 'model = UNetNodeTransformer(' in line:
        j = i
        while ').to(device)' not in lines[j]: 
            j += 1
        patch_code = """
    if getattr(args, 'resume', None) is not None:
        state = torch.load(args.resume, map_location='cpu', weights_only=True)
        state = {k.replace('unet.module.', 'unet.', 1) if k.startswith('unet.') else k: v for k, v in state.items()}
        model.load_state_dict(state, strict=False)
        print(f"\n---> Resumed full model from {args.resume}\n")
"""
        lines.insert(j + 1, patch_code)
        break

with open(script_path, 'w') as f:
    f.writelines(lines)
print("Training script patched to support full model fine-tuning!")


In [ ]:
# Install the tracksdata module which is required by the training script
!pip install -e /content/repo

# Start the Fine-Tuning Process!
# We use a small learning rate (1e-5) for 10 epochs to carefully adapt to the competition test edge cases.
!python /content/repo/scripts/train_unet_transformer.py \
    --data-dir /content/data/train \
    --resume /content/weights/unet_transformer/split_0/checkpoint_last.pth \
    --epochs 10 \
    --lr 1e-5 \
    --batch-size 8 \
    --num-workers 2

# Save the fine-tuned weights permanently to your Google Drive
!mkdir -p /content/drive/MyDrive/biohub_finetuned_weights
!cp -r /content/unet_transformer/* /content/drive/MyDrive/biohub_finetuned_weights/
print("\n✅ Fine-tuning complete! The new weights have been saved to your Google Drive.")
